# Day 49 Revision Summary — Background Jobs for Slow Inference

- **`BackgroundTasks` returns the response first, then runs extra work in the same process** — a function passed to `add_task` doesn't execute until *after* the response has already gone out, so `/signup-fast` returns `200` in ~0.0s while the "slow email" (`/signup-slow`, 2.006s measured) finishes afterward with nobody waiting on it.
- **Multiple `add_task` calls queue in order and all run after the response, in that order** — `/checkout`'s three stacked tasks (charge card -> update inventory -> send receipt email) always logged in the order they were added, confirming FastAPI doesn't reorder or parallelize them.
- **A plain (non-`async def`) function handed to `BackgroundTasks` is dispatched to a worker thread, not run on the event loop** — that's why a 3-second `time.sleep` inside `run_classification_job` doesn't freeze the server: `/health` answered instantly while a classification job was mid-flight.
- **The submit -> `job_id` -> poll pattern decouples "how long the work takes" from "how long the HTTP connection stays open"** — `POST /classify-async` returns a `job_id` in milliseconds with status `"queued"`; the client polls `GET /jobs/{job_id}` on its own schedule until status reaches `"done"` with the real CNN result attached.
- **The in-memory `JOBS` dict and the in-memory rate limiter share the same two failure modes** — neither survives a restart, and Day 47's `--workers N` means each worker process gets its *own* empty dict/counter, so a job or a rate-limit count can go "missing" depending on which worker handles which request; real systems move both into Redis or a database.

## Today's material (Day 49)

Checked all three sources under the `day49` Drive folder:

- **Slides** (`slides/week11_day5_background_jobs.html`) — 26-slide deck covering `BackgroundTasks` basics (Part 1), the submit/poll job pattern wrapped around Day 48's real CNN (Part 2), and a sliding-window rate limiter guarding the new endpoint (Part 3). The deck's own homework-recap slide lists the same tasks as the separate homework file below.
- **Classwork** (`classwork/background_basics.py`, `classwork/async_classify_api.py`, plus `cnn_digit_model.npz` and `sample_images/` reused unchanged from Day 48) — three exercises: `BackgroundTasks` fundamentals, the real job-polling API, and rate limiting `/classify-async`.
- **Homework** (`homework/homework.md`, ~2 hours self-study) — six numbered tasks (with 2b and 5b sub-items): finish two stretch goals left over from class, submit concurrent jobs, write short explanations of the threadpool and multi-worker behavior, and read `formulas.md` before committing.

All three sources agree on what's due today — the homework is mostly "finish the stretch goals from class, then explain *why* three things work the way they do."

## Assignment — Classwork exercises

### Exercise 1 — `BackgroundTasks` basics (`classwork/background_basics.py`, ~20 min)

**What's being asked:** confirm, by direct measurement, that a route with a background task returns before the slow work finishes, and that stacked background tasks run in the order they were added.

**Approach:**
1. Run `uvicorn background_basics:app --reload --port 8000`.
2. `curl -X POST "http://localhost:8000/signup-slow?email=test@example.com"` and note the ~2s wait.
3. `curl -X POST "http://localhost:8000/signup-fast?email=test2@example.com"` and note it returns immediately.
4. Watch the server terminal — confirm `BACKGROUND DONE` prints *after* the fast route's response already went out.
5. Call `/checkout?order_id=ORD1` a few times and confirm the three `STEP` log lines always print in the same order.
6. **Stretch:** add `/signup-fast-multi`, scheduling two background tasks (notify + a fake analytics log call).

### Exercise 2 — Job-polling pattern (`classwork/async_classify_api.py`, ~25 min)

**What's being asked:** stand up the submit -> `job_id` -> poll pattern around Day 48's real CNN and watch a job's full lifecycle, including proof the server stays responsive while a job runs.

**Approach:**
1. Run `uvicorn async_classify_api:app --port 8000` and open `/docs`.
2. `curl -X POST http://localhost:8000/classify-async -F "file=@sample_images/digit_0_label_0.png"` and note the returned `job_id`.
3. Poll `curl http://localhost:8000/jobs/<job_id>` a few times, a second or two apart, and watch `status` move `queued` -> `running` -> `done` with a real `predicted_digit`/`confidence` attached.
4. While a job is mid-flight, call `/health` from a second terminal and confirm it answers instantly.
5. Call `/jobs/does-not-exist` and confirm the `404`.
6. **Stretch:** add `DELETE /jobs/{job_id}`.

### Exercise 3 — Rate limiting `/classify-async` (~15 min)

**What's being asked:** verify the sliding-window limiter actually caps request rate, returns a real `429` with `Retry-After`, and resets once the window clears.

**Approach:**
1. With the server still running, submit 3 sample images back-to-back and confirm all three return `200`.
2. Submit a 4th immediately and confirm a `429` with a `Retry-After` header.
3. `curl http://localhost:8000/rate-limit-status` to see the current count against the limit.
4. Wait 10 seconds and submit again — confirm it now succeeds.
5. **Stretch:** change `check_rate_limit` to key on an API-key header (Day 47 style) instead of `request.client.host`.

## Assignment — Homework (~2 hours self-study)

**1. Finish the multi-task stretch** — *What's asked:* if not done in class, add `/signup-fast-multi` to `background_basics.py`, scheduling two background tasks in one route. *Approach:* copy the `/signup-fast` route, call `background_tasks.add_task` twice (once for `slow_notify`, once for a new fake `log_analytics` function), and confirm both run, in order, after the response.

**2. Finish the job-management stretch** — *What's asked:* add `DELETE /jobs/{job_id}` to `async_classify_api.py`, returning `404` if the job doesn't exist. *Approach:* add a route that does `job = JOBS.pop(job_id, None)`, raises `HTTPException(404, ...)` if `job` is `None`, otherwise returns a confirmation; test against both a real job id and a bogus one.

**2b. Rate-limit by API key, not IP** — *What's asked:* key `request_log` on an `X-API-Key` header (Day 47's pattern) instead of `request.client.host`, and explain in one sentence why this is fairer to clients sharing one public IP. *Approach:* change `check_rate_limit` to read `request.headers.get("X-API-Key", "anonymous")` as the dict key instead of `request.client.host`; the one-sentence answer is that IP-based limiting punishes every device behind a shared IP (an office or campus NAT) for one noisy neighbor's usage, while an API key identifies the actual client regardless of how many others share its network.

**3. Submit several jobs at once** — *What's asked:* submit 3 different sample images to `/classify-async` back-to-back without waiting, record all 3 `job_id`s, then poll all 3 and confirm each independently reaches `"done"` with its own correct result. *Approach:* fire three `curl -F` submissions in a row, save each `job_id`, then loop `GET /jobs/{job_id}` for each until all three report `"done"`, and check each `predicted_digit` matches its own uploaded image.

**4. Explain the threadpool behavior (3 sentences)** — *What's asked:* why does `/health` answer instantly while a classification job is running, and what would change if `run_classification_job` used `asyncio` directly instead of `time.sleep`? *Approach to answering:* explain that FastAPI runs plain sync background functions in a worker thread pool rather than on the event loop, so a blocking `time.sleep` only occupies that thread, not the loop handling other requests; note that switching to a genuinely `async def` job with `await asyncio.sleep(...)` would run it back on the event loop, where a long-running synchronous section (unlike a proper `await`) could still block other requests if not written carefully.

**5. Explain the multi-worker gap (3 sentences)** — *What's asked:* why might `GET /jobs/{job_id}` 404 immediately after a successful submit when the server runs with `--workers 4`, and what fixes it? *Approach to answering:* note that `--workers 4` starts four separate OS processes, each with its own empty `JOBS` dict in its own memory; the submit and the poll can land on different worker processes (load-balanced independently), so the poll's process never saw the job that only exists in another process's dict; the fix is moving the job store to something shared across processes, like Redis or a database.

**5b. Explain the rate limiter's own multi-worker gap (2 sentences)** — *What's asked:* why could a client get up to 4x the intended limit under `--workers 4`, and what's the fix? *Approach to answering:* each of the 4 worker processes keeps its own `request_log` dict, so a client's requests spread across workers are counted separately by each one, letting up to `4 x RATE_LIMIT_MAX_REQUESTS` through in the same window; the fix is the same as #5 — move the counters into a shared store like Redis so all workers see one count.

**6. Read + commit** — *What's asked:* read `formulas.md` end-to-end (BackgroundTasks mechanics, task ordering, the threadpool explanation, the submit/poll pattern, the in-memory store's real-world limitations) and commit the day's work. *Approach:* read the doc, then `git add . && git commit -m "week11 day5 — background jobs for slow inference"`.